In [1]:
#### --------------------------------------------------------------------------------------------------------------
#### author: Ranjan Barman, date: May 30, 2025 (modified for immune only)
#### Compute molecular features (immune nuclei only) from Hover-Net predicted json files (one slide)
#### Saves output in features3 folder
#### --------------------------------------------------------------------------------------------------------------

import os
import json
import cv2
import numpy as np
import pandas as pd
from shapely.geometry import Polygon
from argparse import ArgumentParser

# Set working directory
_wpath_ = "/data/Lab_ruppin/Ranjan/HnE/"
os.makedirs(_wpath_, exist_ok=True)
os.chdir(_wpath_)
print("Working directory:", _wpath_)

# Dataset name
dataset_name = "TCGA_BRCA_FFPE"

# Parse slide argument
parser = ArgumentParser()
parser.add_argument("-slide", type=str, required=True, help="Slide folder name to process")
args = parser.parse_args()
slide_folder = args.slide

# Define the base directory containing all slides
hovernet_base_dir = f"{dataset_name}/outputs/HoverNet/"
slide_path = os.path.join(hovernet_base_dir, slide_folder)

if not os.path.isdir(slide_path):
    print(f"Error: {slide_folder} is not a valid directory.")
    exit(1)

tiles_dir = os.path.join(slide_path, "tiles")
json_dir = os.path.join(slide_path, "masks/json")
overlay_dir = os.path.join(slide_path, "masks/overlay")
features_dir = os.path.join(slide_path, "features3")  # Changed to features3

os.makedirs(features_dir, exist_ok=True)

# Output file for this slide
output_file = os.path.join(features_dir, f"{slide_folder}.csv")

# List all JSON files in the masks/json directory, sorted in numerical order
if not os.path.exists(json_dir):
    print(f"Skipping {slide_folder}: No JSON directory found.")
    exit(1)

json_files = sorted([f for f in os.listdir(json_dir) if f.endswith(".json")], 
                    key=lambda x: int(x.split("_")[-1].split(".")[0]))

# Process each tile
results = []
for json_file in json_files:
    json_path = os.path.join(json_dir, json_file)
    tile_name = json_file.replace(".json", "")
    
    # Read JSON data
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extract immune nuclei only (type = 2)
    nuclei = data.get('nuc', {})
    immune_nuclei = {key: val for key, val in nuclei.items() if val['type'] == 2}

    for key, nucleus in immune_nuclei.items():
        nucleus_id = key
        contour = np.array(nucleus['contour'])
        polygon = Polygon(contour)
        area = polygon.area
        perimeter = polygon.length

        # Fit ellipse
        if len(contour) >= 5:
            ellipse = cv2.fitEllipse(contour)
            major_axis = max(ellipse[1])
            minor_axis = min(ellipse[1])
        else:
            major_axis = minor_axis = 0

        # Eccentricity
        if major_axis > 0:
            eccentricity = np.sqrt(1 - (minor_axis**2 / major_axis**2))
        else:
            eccentricity = 0

        # Circularity
        if perimeter > 0:
            circularity = (4 * np.pi * area) / (perimeter ** 2)
        else:
            circularity = 0

        # Save result
        results.append([
            slide_folder, tile_name, nucleus_id, "Immune",
            area, major_axis, minor_axis, perimeter,
            eccentricity, circularity
        ])

# Convert to DataFrame
columns = ["Slide", "Tile", "Nucleus ID", "Nucleus Type", "Area", "Major Axis", "Minor Axis", "Perimeter", "Eccentricity", "Circularity"]
df_results = pd.DataFrame(results, columns=columns)

# Save to CSV
df_results.to_csv(output_file, mode='w', index=False, header=True)

print(f"Immune nuclear morphology features saved for slide {slide_folder} to {output_file}")


Working directory: /data/Lab_ruppin/Ranjan/HnE/


usage: ipykernel_launcher.py [-h] -slide SLIDE
ipykernel_launcher.py: error: the following arguments are required: -slide


SystemExit: 2

/usr/local/Anaconda/envs/py3.10/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
